# Sorting

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Trace merge sort's merges and explain why it takes O(n log n)
- Write the merge step, and reuse it to count inversions
- Sort by a key with a comparator, then sweep the sorted data in one pass
- Use counting sort when the values fall in a small range

## In short

Sorting makes data easy to search, group and compare, and the main O(n log n) sorts use one of two ideas.

**Merge sort** merges sorted runs. Merging two sorted lists takes one comparison per value taken, so a pass over n values costs about n. Start with runs of 1 and double their size every pass: after log₂ n passes there is one sorted run, for O(n log n) in total, always.

![Merge sort's passes on 8 values](passes.svg)

**O(n²) against O(n log n):** selection sort and `Arrays.sort` on the same random numbers. Double `n` and compare.

In [ ]:
int n = 20_000;
int[] a = new Random(2).ints(n, 0, 1_000_000).toArray();
int[] b = a.clone();

long start = System.nanoTime();
for (int i = 0; i < n; i++) { // selection sort
    int min = i;
    for (int j = i + 1; j < n; j++) {
        if (a[j] < a[min]) {
            min = j;
        }
    }
    int t = a[i];
    a[i] = a[min];
    a[min] = t;
}
long middle = System.nanoTime();
Arrays.sort(b);
long end = System.nanoTime();

IO.println("selection sort: " + (middle - start) / 1_000_000 + " ms");
IO.println("Arrays.sort:    " + (end - middle) / 1_000_000 + " ms");
IO.println("same result: " + Arrays.equals(a, b));

**Quicksort** picks a pivot, moves smaller values to its left and bigger ones to its right (a **partition**), then sorts each side. It is O(n log n) on average, but a bad pivot every time, such as the first element of a sorted array, makes it O(n²).

**A bad pivot every time.** This quicksort always picks the first element. Count its comparisons on random and on already sorted data.

In [ ]:
// shared: quicksort with the first element as the pivot, counting comparisons
long comparisons = 0;

void quicksort(int[] a, int lo, int hi) {
    if (lo >= hi) {
        return;
    }
    int pivot = a[lo];
    int i = lo;
    for (int j = lo + 1; j <= hi; j++) {
        comparisons++;
        if (a[j] < pivot) {
            i++;
            int t = a[i];
            a[i] = a[j];
            a[j] = t;
        }
    }
    int t = a[lo];
    a[lo] = a[i];
    a[i] = t;
    quicksort(a, lo, i - 1);
    quicksort(a, i + 1, hi);
}

In [ ]:
int n = 3000;
int[] random = new Random(3).ints(n, 0, 1_000_000).toArray();
quicksort(random, 0, n - 1);
IO.println(String.format("random: %,d comparisons", comparisons));

comparisons = 0;
int[] sorted = new int[n];
for (int i = 0; i < n; i++) {
    sorted[i] = i;
}
quicksort(sorted, 0, n - 1);
IO.println(String.format("sorted: %,d comparisons (n²/2 = %,d)", comparisons, (long) n * n / 2));

When the values are small whole numbers, such as scores from 0 to 100, **counting sort** doesn't compare at all: it counts how often each value appears, in O(n + k) for k possible values.

In Java, `Arrays.sort` on an `int[]` uses a quicksort, and on objects a merge sort driven by a **comparator**, such as `Comparator.comparingInt(Meeting::start)`. That sort is **stable**: equal elements keep their order.

Watch out: a comparator written as `a - b` can overflow. Use `Integer.compare(a, b)`.

**Comparators:** by length, then alphabetically, and reversed. The last line shows the `a - b` overflow.

In [ ]:
List<String> words = new ArrayList<>(List.of("pear", "fig", "apple", "kiwi", "date"));
words.sort(Comparator.comparingInt(String::length).thenComparing(Comparator.naturalOrder()));
IO.println(words);
words.sort(Comparator.comparingInt(String::length).reversed());
IO.println(words);

List<Integer> numbers = new ArrayList<>(List.of(2_000_000_000, -1_000_000_000, 0));
numbers.sort((a, b) -> a - b); // overflows
IO.println("with a - b:            " + numbers);
numbers.sort(Integer::compare);
IO.println("with Integer.compare:  " + numbers);

## Readings

- [Sorting](https://visualgo.net/en/sorting) (VisuAlgo (National University of Singapore), 20 min): Pick Merge Sort, Quick Sort and Counting Sort from the top menu and run each on your own short array. Watch the merge take the smaller front value, the partition move values around the pivot, and counting sort never compare two values.
- [Writing and Combining Comparators](https://dev.java/learn/language/fp/lambdas/writing-comparators/) (dev.java (Oracle), 8 min): Comparator.comparing and comparingInt build a comparator from a key, thenComparing breaks ties with a second key, and reversed() flips the order. Note why (i1 - i2) is a bad comparator.

## Check yourself

1. How many passes does bottom-up merge sort make on 1,000 values?
2. Why can't counting sort sort names?
3. What does a stable sort keep, and when does that matter?

## Challenges

Solve the unit's challenges right here: each cell starts with `// sphinx: challenge …`, and running it runs **all** of the challenge's tests, hidden ones included. A pass counts in your progress in the Sphinx sidebar, just like Submit. The full description, with examples and hints, opens from the sidebar.

### Exam Scores · Easy

In [ ]:
// sphinx: challenge exam-scores
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] scores = new int[n];
    for (int i = 0; i < n; i++) {
        scores[i] = Integer.parseInt(parts[i]);
    }

    // TODO: count each score, print the counts from 100 down to 0, then the median
}

### Merge Trace · Medium

In [ ]:
// sphinx: challenge merge-trace
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] a = new int[n];
    for (int i = 0; i < n; i++) {
        a[i] = Integer.parseInt(parts[i]);
    }

    // TODO: merge runs of width 1, 2, 4, ..., print the array after each pass, then the comparisons
}

### Calendar Merge · Medium

In [ ]:
// sphinx: challenge calendar-merge
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    int[][] meetings = new int[n][2];
    for (int i = 0; i < n; i++) {
        String[] parts = IO.readln().trim().split(" +");
        meetings[i][0] = Integer.parseInt(parts[0]);
        meetings[i][1] = meetings[i][0] + Integer.parseInt(parts[1]); // the end
    }

    // TODO: sort the meetings by start, join overlapping ones into blocks, and print the three lines
}

### Ranking Distance · Hard

In [ ]:
// sphinx: challenge ranking-distance
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] ranking = new int[n];
    for (int i = 0; i < n; i++) {
        ranking[i] = Integer.parseInt(parts[i]);
    }

    // TODO: count the inversions with a merge sort, then print them and the agreement
}